In [1]:
from fast_borf.weighted.iborf import IBORF
import xarray as xr
import numpy as np

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import f1_score

In [3]:
from irregular_ts.data_utils import data_new_folder
import xarray as xr
df = xr.open_dataset(data_new_folder() / "Abf.h5", engine="my_engine")["data"]
y, split = df.irr.get_task_target_and_split()
X, _ = df.irr.to_dense(
    concatenate_time=True,
    normalize_time=True,
)
train_idxs, test_idxs = split == "train", split == "test"
X_train, y_train = X[train_idxs], y[train_idxs]
X_test, y_test = X[test_idxs], y[test_idxs]
X_train.shape

(30, 2, 128)

In [4]:
from sklearn.linear_model import RidgeClassifierCV
from sklearn.utils.extmath import softmax
import numpy as np


class RidgeClassifierCVFix(RidgeClassifierCV):

    # def predict_proba(self, X):
    #     return np.eye(len(self.classes_))[self.predict(X)]

    def predict_proba(self, X):
        d = self.decision_function(X)
        if len(d.shape) == 1:
            d = np.c_[-d, d]
        return softmax(d)

    # d = clf.decision_function(x)[0]
    # probs = np.exp(d) / np.sum(np.exp(d))

In [5]:
borf = IBORF(
    contains_time_idx=True,
    min_window_to_signal_std_ratio=0.15,
    n_jobs=-1
)

In [6]:
pipe = make_pipeline(
    borf,
    FunctionTransformer(lambda x: np.arcsinh(x)),
    RidgeClassifierCVFix()
)

In [7]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
f1_score(y_test, y_pred, average="macro")

/Users/francesco/miniforge3/envs/timeseries_dl/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/francesco/miniforge3/envs/timeseries_dl/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/francesco/miniforge3/envs/timeseries_dl/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warning

0.8951712319329848

In [8]:
pipe.predict_proba(X_test)

/Users/francesco/miniforge3/envs/timeseries_dl/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/francesco/miniforge3/envs/timeseries_dl/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/francesco/miniforge3/envs/timeseries_dl/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warning

array([[0.38642146, 0.37121972, 0.24235882],
       [0.34812293, 0.40999179, 0.24188528],
       [0.44558118, 0.22140675, 0.33301207],
       ...,
       [0.22454511, 0.11107374, 0.66438115],
       [0.28605422, 0.17829957, 0.53564621],
       [0.2952185 , 0.17082746, 0.53395404]])